In [1]:
import glob
import os
import pandas as pd
from collections import defaultdict
import numpy as np
from tqdm import tqdm

In [2]:
INPUT_DIR = "../mbr_studies_2year"
OUTPUT_DIR = "../mbr_studies_2year_filtered_traintest"
os.makedirs(INPUT_DIR, exist_ok=True)

OK_LIST_PATH = os.path.join(INPUT_DIR, "ok_files.txt")
NOT_OK_LIST_PATH = os.path.join(INPUT_DIR, "not_ok_files.txt")
BAD_INDIVIDUALS_PATH = os.path.join(INPUT_DIR, "bad_individuals.txt")
SPLIT_OVERRIDES_PATH = os.path.join(INPUT_DIR, "split_overrides.txt")

In [3]:
# all_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_2year.csv")))
# all_files
def _read_list(path: str) -> set:
    """Read a newline-separated list of filenames (basenames)."""
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return set(lines)

def get_review_status():
    ok = _read_list(OK_LIST_PATH)
    not_ok = _read_list(NOT_OK_LIST_PATH)
    return ok, not_ok

def get_remaining_files():
    """Return (remaining_files, ok_set, not_ok_set)."""
    all_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_2year.csv")))
    ok_set, not_ok_set = get_review_status()
    reviewed_basenames = ok_set | not_ok_set

    remaining = [f for f in all_files
                 if os.path.basename(f) not in reviewed_basenames]
    return remaining, ok_set, not_ok_set

remaining_files, ok_set, not_ok_set = get_remaining_files()
all_files = list(ok_set)
len(all_files)

40

In [4]:
def _read_bad_individuals() -> set:
    """
    Read bad individuals from BAD_INDIVIDUALS_PATH.

    Each line: <csv_basename>,<individual-local-identifier>
    Returns a set of (filename, individual_id) tuples.
    """
    if not os.path.exists(BAD_INDIVIDUALS_PATH):
        return set()

    bad = set()
    with open(BAD_INDIVIDUALS_PATH, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = ln.split(",", 1)
            if len(parts) == 2:
                fname, indiv = parts
                bad.add((fname, indiv))
    return bad

def _read_split_overrides() -> dict:
    """
    Read split overrides from SPLIT_OVERRIDES_PATH.

    Each line: <csv_basename>,<individual-local-identifier>,YYYY-MM-DD
    Returns a dict mapping (filename, individual_id) -> pd.Timestamp
    (date at midnight).
    """
    overrides: dict[tuple[str, str], pd.Timestamp] = {}
    if not os.path.exists(SPLIT_OVERRIDES_PATH):
        return overrides

    with open(SPLIT_OVERRIDES_PATH, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = ln.split(",", 2)
            if len(parts) != 3:
                continue
            fname, indiv, date_str = parts
            try:
                ts = pd.to_datetime(date_str)
            except Exception:
                continue
            overrides[(fname, indiv)] = ts
    return overrides


In [ ]:
for csv_path in tqdm(all_files):
    df_full = pd.read_csv(os.path.join(INPUT_DIR,csv_path))
    df_full['individual-local-identifier'] = df_full['individual-local-identifier'].astype(str)

    basename = os.path.basename(csv_path)
    study_name = basename.replace("_2year.csv", "")

    # 2) Apply bad-individual filtering for the current view
    bad_indivs = _read_bad_individuals()
    bad_here = [ ind for (fname, ind) in bad_indivs if fname == basename ]
    # print("Bad here?", bad_here, df_full[df_full['individual-local-identifier'].astype(str).isin(bad_here)])
    if bad_here:
        df_filtered = df_full[~df_full['individual-local-identifier'].astype(str).isin(bad_here)].copy()
    else:
        df_filtered = df_full.copy()

    if df_filtered.empty:
        raise RuntimeError(
            "All individuals in this file are marked bad (or no data left); nothing to plot."
        )

    # 3) Apply split overrides for this file
    all_split_overrides = _read_split_overrides()
    overrides_here: dict[str, pd.Timestamp] = {
        indiv: ts
        for (fname, indiv), ts in all_split_overrides.items()
        if fname == basename
    }

    df = df_filtered
    split_times = overrides_here if overrides_here else None

    if 'timestamp' not in df.columns:
        raise RuntimeError("Expected a 'timestamp' column in the data.")

    df = df.copy()
    df['timestamp'] = pd.to_datetime(
        df['timestamp'],
            format='mixed',   # let pandas infer per element
            errors='coerce'   # bad parses become NaT instead of raising
        )
    df = df[df['timestamp'].notna()].copy()
    df['individual-local-identifier'] = df['individual-local-identifier'].astype(str)

    duration = df['timestamp'].max() - df['timestamp'].min()
    num_individuals = df['individual-local-identifier'].nunique()
    num_points = len(df)

    if 'individual-taxon-canonical-name' in df.columns:
        species = df['individual-taxon-canonical-name'].dropna().unique()
        species_str = ", ".join(map(str, species))
    else:
        species_str = "N/A (column 'individual-taxon-canonical-name' missing)"

    # For the UI: counts
    individual_counts = df['individual-local-identifier'].value_counts().sort_index()

    

    # Timeline prep
    df_timeline = df.copy()
    df_timeline = df_timeline.sort_values(['individual-local-identifier', 'timestamp'])
    gap_threshold = pd.Timedelta(days=3)

    span_time = (
        df_timeline.groupby('individual-local-identifier')['timestamp']
        .agg(start='min', end='max')
    )
    span_time['default_mid'] = span_time['start'] + (span_time['end'] - span_time['start']) / 2

    # Tracks & maps prep
    df_tracks = df.copy()
    df_tracks = df_tracks.sort_values(['individual-local-identifier', 'timestamp'])

    span_tracks = (
        df_tracks.groupby('individual-local-identifier')['timestamp']
        .agg(start='min', end='max')
    )
    span_tracks['default_mid'] = span_tracks['start'] + (span_tracks['end'] - span_tracks['start']) / 2

    # -------------------------------------------------------------------
    # Apply split overrides
    # -------------------------------------------------------------------
    # Default: use mid-time
    split_time_series = span_time['default_mid'].copy()
    split_time_tracks = span_tracks['default_mid'].copy()

    if split_times is not None:
        for indiv, ts in split_times.items():
            if indiv not in span_time.index:
                continue
            # Clip to individual range (timeline)
            start = span_time.loc[indiv, 'start']
            end = span_time.loc[indiv, 'end']
            clipped = min(max(ts, start), end)
            split_time_series.loc[indiv] = clipped

            if indiv in span_tracks.index:
                start_t = span_tracks.loc[indiv, 'start']
                end_t = span_tracks.loc[indiv, 'end']
                clipped_t = min(max(ts, start_t), end_t)
                split_time_tracks.loc[indiv] = clipped_t

    span_time['split'] = split_time_series
    span_tracks['split'] = split_time_tracks

    individuals_timeline = span_time.index.tolist()

    df_tracks['split_time'] = df_tracks['individual-local-identifier'].map(span_tracks['split'])
    df_tracks['set'] = np.where(
        df_tracks['timestamp'] <= df_tracks['split_time'],
        'train',
        'test'
    )

    if len(df) > 600_000:
        print("Not writing, too big")
    elif len(df) == 0:
        print("Not writing, it's empty")
    else:
        print("Writing with length", len(df))
        df_tracks.to_csv(os.path.join(OUTPUT_DIR, os.path.basename(csv_path).replace(".csv", "_filtered_traintest.csv")))

  0%|          | 0/40 [00:00<?, ?it/s]

Writing with length 112649


  2%|▎         | 1/40 [00:00<00:35,  1.11it/s]

Writing with length 6198


/tmp/ipykernel_720746/1797594752.py:2: DtypeWarning: Columns (18,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv(os.path.join(INPUT_DIR,csv_path))
  8%|▊         | 3/40 [00:03<00:43,  1.17s/it]

Not writing, too big
Writing with length 31360


... (50 outputs left out)

 95%|█████████▌| 38/40 [00:40<00:03,  1.66s/it]

Writing with length 122180


100%|██████████| 40/40 [00:41<00:00,  1.03s/it]

Writing with length 21397


In [7]:
df.head()

,event-id,visible,timestamp,location-long,location-lat,algorithm-marked-outlier,external-temperature,gps:hdop,gps:maximum-signal-strength,gps:satellite-count,height-above-msl,manually-marked-outlier,sensor-type,individual-taxon-canonical-name,tag-local-identifier,individual-local-identifier,study-name
0,5289850520,True,2016-07-31 14:00:36,141.072208,-33.194244,NaN,29.25,2.1,42.0,5,72.0,NaN,gps,Vulpes vulpes,150-5573,Flo,"Red Fox (Vulpes vulpes) - Scotia, NSW, Australia"
1,5289850521,True,2016-07-31 14:01:06,141.072222,-33.194231,NaN,28.68,1.9,43.0,6,66.0,NaN,gps,Vulpes vulpes,150-5573,Flo,"Red Fox (Vulpes vulpes) - Scotia, NSW, Australia"
2,5289850522,True,2016-07-31 14:20:37,141.072228,-33.194239,NaN,28.75,2.6,42.0,5,66.0,NaN,gps,Vulpes vulpes,150-5573,Flo,"Red Fox (Vulpes vulpes) - Scotia, NSW, Australia"
3,5289850523,True,2016-07-31 14:40:37,141.072075,-33.194325,NaN,28.62,3.8,39.0,5,58.0,NaN,gps,Vulpes vulpes,150-5573,Flo,"Red Fox (Vulpes vulpes) - Scotia, NSW, Australia"
4,5289850524,True,2016-07-31 15:00:38,141.072197,-33.194231,NaN,29.31,4.1,37.0,5,65.0,NaN,gps,Vulpes vulpes,150-5573,Flo,"Red Fox (Vulpes vulpes) - Scotia, NSW, Australia"


----

Make sure it's ok

In [11]:
indivs = 0
pts = 0
for study in tqdm(glob.glob(OUTPUT_DIR + "/*.csv")):
    _df = pd.read_csv(study)
    indivs += len(_df['individual-local-identifier'].astype(str).unique())
    pts += len(_df)

indivs, pts

  0%|          | 0/37 [00:00<?, ?it/s]/tmp/ipykernel_720746/3635347375.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  _df = pd.read_csv(study)
 62%|██████▏   | 23/37 [00:02<00:00, 18.52it/s]/tmp/ipykernel_720746/3635347375.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  _df = pd.read_csv(study)
 70%|███████   | 26/37 [00:02<00:00, 14.82it/s]/tmp/ipykernel_720746/3635347375.py:4: DtypeWarning: Columns (6,15,16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  _df = pd.read_csv(study)
 76%|███████▌  | 28/37 [00:03<00:01,  7.14it/s]/tmp/ipykernel_720746/3635347375.py:4: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  _df = pd.read_csv(study)
 81%|████████  | 30/37 [00:03<00:00,  7.79it/s]/tmp/ipykernel_720746/3635347375.py:4: DtypeWarning: Columns (19) have mixed types. Specify dtype op

(385, 3195643)